# Crawling web detik.com

In [1]:
import sys

print(sys.executable)

C:\Users\javier\Downloads\PPW\enWebmining\Scripts\python.exe


In [2]:
import trafilatura

print("Trafilatura berhasil digunakan")

Trafilatura berhasil digunakan


In [4]:
pip install requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.


In [17]:
import requests
import pandas as pd
import trafilatura
import time
import os

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

In [18]:
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en;q=0.8"
})

print("Session siap")

Session siap


### Fungsi mengambil halaman

In [19]:
def get_html(url):
    try:
        response = session.get(
            url,
            timeout=20
        )

        response.raise_for_status()

        return response.text

    except requests.RequestException as e:
        print(f"Gagal mengambil halaman: {url}")
        print(f"Error: {e}")
        return None

### Tes koneksi ke detik sport

In [20]:
sport_home = "https://sport.detik.com/"

html = get_html(sport_home)

if html:
    print("Berhasil mengambil halaman Sport")
    print("Panjang HTML:", len(html))
else:
    print("Gagal mengambil halaman Sport")

Berhasil mengambil halaman Sport
Panjang HTML: 268626


### Tes koneksi finance

In [21]:
finance_home = "https://finance.detik.com/"

html_finance = get_html(finance_home)

if html_finance:
    print("Berhasil mengambil halaman Finance")
    print("Panjang HTML:", len(html_finance))
else:
    print("Gagal mengambil halaman Finance")

Berhasil mengambil halaman Finance
Panjang HTML: 340271


### Fungsi mengambil URL artikel

In [22]:
def get_article_links(page_url, domain):
    html = get_html(page_url)

    if not html:
        return []

    soup = BeautifulSoup(html, "lxml")

    article_links = set()

    for a in soup.find_all("a", href=True):

        href = a["href"].strip()

        # Mengubah URL relatif menjadi URL lengkap
        full_url = urljoin(page_url, href)

        parsed = urlparse(full_url)

        # Hanya mengambil domain yang sesuai
        if parsed.netloc != domain:
            continue

        # Hilangkan fragment
        full_url = full_url.split("#")[0]

        # Hanya HTTP/HTTPS
        if parsed.scheme not in ["http", "https"]:
            continue

        article_links.add(full_url)

    return list(article_links)

### Tes mengambil URL Sport

In [23]:
sport_urls_test = get_article_links(
    "https://sport.detik.com/",
    "sport.detik.com"
)

print("Jumlah URL ditemukan:", len(sport_urls_test))

for url in sport_urls_test[:10]:
    print(url)

Jumlah URL ditemukan: 73
https://sport.detik.com/sepakbola/uefa/d-8655331/luis-enrique-nggak-mau-ingat-ingat-psg-juara-bertahan-liga-champions
https://sport.detik.com/sepakbola/uefa/d-8655270/yamal-cuma-kurang-trofi-liga-champions-bertekad-juara-di-musim-ini
https://sport.detik.com
https://sport.detik.com/
https://sport.detik.com/sport-lain/d-8655872/pelatih-timnas-voli-putra-ri-targetkan-medali-di-asian-games-2026
https://sport.detik.com/sepakbola/uefa/d-8655901/enzo-fernandez-sanjung-haaland-siap-layani-dengan-banyak-assist
https://sport.detik.com/sepakbola/bola-dunia/d-8655896/jangan-kaget-kalau-mbappe-raih-ballon-dor-2026
https://sport.detik.com/sepakbola/liga-indonesia/d-8655279/fix-persija-menjamu-persib-di-sugbk
https://sport.detik.com/bold-riders
https://sport.detik.com/sepakbola/liga-indonesia/d-8656106/sjh-disiapkan-buat-fifa-asean-cup-persib-terpaksa-main-di-gbla


### Filter URL artikel Sport

In [24]:
def is_sport_article(url):
    parsed = urlparse(url)

    if parsed.netloc != "sport.detik.com":
        return False

    path = parsed.path.lower()

    # URL umum yang bukan artikel
    excluded = [
        "/",
        "/index",
        "/indeks",
        "/search",
        "/video",
        "/foto"
    ]

    if path in excluded:
        return False

    # Hindari URL folder/kategori
    if path.endswith("/"):
        return False

    return True

In [25]:
sport_article_urls = [
    url
    for url in sport_urls_test
    if is_sport_article(url)
]

sport_article_urls = list(dict.fromkeys(sport_article_urls))

print(
    "URL artikel Sport:",
    len(sport_article_urls)
)

URL artikel Sport: 68


### Cek URL Sport

In [26]:
for url in sport_article_urls[:20]:
    print(url)

https://sport.detik.com/sepakbola/uefa/d-8655331/luis-enrique-nggak-mau-ingat-ingat-psg-juara-bertahan-liga-champions
https://sport.detik.com/sepakbola/uefa/d-8655270/yamal-cuma-kurang-trofi-liga-champions-bertekad-juara-di-musim-ini
https://sport.detik.com
https://sport.detik.com/sport-lain/d-8655872/pelatih-timnas-voli-putra-ri-targetkan-medali-di-asian-games-2026
https://sport.detik.com/sepakbola/uefa/d-8655901/enzo-fernandez-sanjung-haaland-siap-layani-dengan-banyak-assist
https://sport.detik.com/sepakbola/bola-dunia/d-8655896/jangan-kaget-kalau-mbappe-raih-ballon-dor-2026
https://sport.detik.com/sepakbola/liga-indonesia/d-8655279/fix-persija-menjamu-persib-di-sugbk
https://sport.detik.com/bold-riders
https://sport.detik.com/sepakbola/liga-indonesia/d-8656106/sjh-disiapkan-buat-fifa-asean-cup-persib-terpaksa-main-di-gbla
https://sport.detik.com/sepakbola/liga-indonesia/d-8655338/head-to-head-persija-vs-persib-10-laga-terbaru-macan-kemayoran-kurang-oke
https://sport.detik.com/sport-

### Membuat fungsi ekstraksi artikel

In [27]:
def extract_article(url):
    try:
        html = get_html(url)

        if not html:
            return None

        text = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False,
            favor_precision=True
        )

        if not text:
            return None

        text = text.strip()

        if len(text) < 200:
            return None

        return text

    except Exception as e:
        print(f"Gagal ekstraksi: {url}")
        print(e)
        return None

### Tes satu artikel

In [28]:
test_url = sport_article_urls[0]

print(test_url)

https://sport.detik.com/sepakbola/uefa/d-8655331/luis-enrique-nggak-mau-ingat-ingat-psg-juara-bertahan-liga-champions


In [29]:
text = extract_article(test_url)

if text:
    print(text[:3000])
else:
    print("Isi artikel tidak ditemukan")

Paris Saint-Germain (PSG) adalah juara bertahan Liga Champions. Status itu nggak mau diingat-ingat pelatih Luis Enrique!
PSG dua kali menangi Liga Champions di dua musim terakhir. Inter Milan dan Arsenal jadi korban di laga final.
PSG tentu kembali jadi favorit untuk jadi kampiun Liga Champions musim 2026/27 ini. Akan tetapi, pelatih Luis Enrique ogah bicarakan kans terlalu jauh dan ogah ingat-ingat status juara bertahan!
"Yang berlalu sudah berlalu," tegas Enrique dilansir dari situs resmi UEFA.
"Sama seperti lisensi berkendara, itu harus diperbarui," sambungnya.
Luis Enrique tidak mau PSG terlena dan di atas angin. Kompetisi Liga Champions selalu berat.
Lawan pertama mereka adalah Slovan Bratislava, yang akan main pada Kamis (10/9) pukul 02.00 WIB. PSG memulai Ligue 1 dengan pincang, Enrique makin peringatkan timnya.
"Semua orang bersemangat untuk bertanding melawan kami. Kami harus menerimanya dan keluar dari zona nyaman," tegasnya.
"Saya siap menunjukkan bahwa timi ini harus terus 

### Membuat crawler artikel

In [30]:
def crawl_articles(urls, label, target=100):
    
    data = []
    visited = set()

    for url in urls:

        if len(data) >= target:
            break

        if url in visited:
            continue

        visited.add(url)

        print(
            f"[{label}] "
            f"{len(data) + 1}/{target} "
            f"-> {url}"
        )

        text = extract_article(url)

        if text is None:
            print("  -> Gagal mendapatkan isi")
            continue

        data.append({
            "id": len(data) + 1,
            "isi_berita": text,
            "label": label
        })

        print("  -> Berhasil")

        # jeda antar-request
        time.sleep(1)

    return data

### Membuat fungsi pagination

In [31]:
def get_sport_index_url(page):
    
    if page == 1:
        return "https://sport.detik.com/indeks"
    
    return f"https://sport.detik.com/indeks?page={page}"

### Tes pagination

In [32]:
for page in range(1, 6):
    print(get_sport_index_url(page))

https://sport.detik.com/indeks
https://sport.detik.com/indeks?page=2
https://sport.detik.com/indeks?page=3
https://sport.detik.com/indeks?page=4
https://sport.detik.com/indeks?page=5


### Mengumpulkan URL Sport

In [33]:
def collect_sport_urls(max_pages=10, target_urls=150):
    
    all_urls = set()

    for page in range(1, max_pages + 1):

        index_url = get_sport_index_url(page)

        print(
            f"Mengambil indeks Sport halaman {page}"
        )

        urls = get_article_links(
            index_url,
            "sport.detik.com"
        )

        for url in urls:

            if is_sport_article(url):
                all_urls.add(url)

        print(
            f"Total URL unik: {len(all_urls)}"
        )

        if len(all_urls) >= target_urls:
            break

        time.sleep(1)

    return list(all_urls)

### Menjalankan pengumpulan Sport

In [34]:
sport_urls = collect_sport_urls(
    max_pages=10,
    target_urls=150
)

Mengambil indeks Sport halaman 1
Total URL unik: 34
Mengambil indeks Sport halaman 2
Total URL unik: 54
Mengambil indeks Sport halaman 3
Total URL unik: 74
Mengambil indeks Sport halaman 4
Total URL unik: 94
Mengambil indeks Sport halaman 5
Total URL unik: 113
Mengambil indeks Sport halaman 6
Total URL unik: 132
Mengambil indeks Sport halaman 7
Total URL unik: 151


In [35]:
print("Total URL Sport:", len(sport_urls))

Total URL Sport: 151


### Crawling 100 Sport

In [36]:
sport_data = crawl_articles(
    sport_urls,
    label="sport",
    target=100
)

[sport] 1/100 -> https://sport.detik.com
  -> Gagal mendapatkan isi
[sport] 1/100 -> https://sport.detik.com/fotosport/d-8639993/obor-menyala-indonesia-siap-berlaga-di-asian-games-2026
  -> Berhasil
[sport] 2/100 -> https://sport.detik.com/sport-lain/d-8649199/wec-2026-paruh-kedua-musim-dimulai-team-wrt-32-lebih-pede
  -> Berhasil
[sport] 3/100 -> https://sport.detik.com/raket/d-8641072/campus-league-badminton-lahirkan-talenta-baru
  -> Berhasil
[sport] 4/100 -> https://sport.detik.com/sport-lain/d-8655779/asian-games-2026-prabowo-janjikan-bonus-rp-3-m-untuk-peraih-emas
  -> Berhasil
[sport] 5/100 -> https://sport.detik.com/sport-lain/d-8648160/pemanasan-timnas-basket-3x3-di-taiwan-jelang-asian-games-2026
  -> Berhasil
[sport] 6/100 -> https://sport.detik.com/raket/d-8641178/seri-keempat-m-15-mens-world-tennis-championship-dimulai
  -> Berhasil
[sport] 7/100 -> https://sport.detik.com/sport-lain/d-8639771/menangi-debut-ufc-bilal-hasan-samai-rekor-jeka-saragih
  -> Berhasil
[sport] 8/10

In [37]:
print("Jumlah data Sport:", len(sport_data))

Jumlah data Sport: 100


In [38]:
sport_df = pd.DataFrame(sport_data)

sport_df.head()

,id,isi_berita,label
0,1,Jakarta - Sebanyak 432 atlet membela Indonesia...,sport
1,2,FIA World Endurance Championship 2026 memasuki...,sport
2,3,Campus League Badminton Regional Semarang tela...,sport
3,4,Kontingen Indonesia bersiap menatap Asian Game...,sport
4,5,"Jelang Asian Games 2026, Timnas basket 3x3 put...",sport


## Finance

### Filter URL artikel finane

In [41]:
def is_finance_article(url):
    parsed = urlparse(url)

    if parsed.netloc != "finance.detik.com":
        return False

    path = parsed.path.lower()

    excluded = [
        "/",
        "/index",
        "/indeks",
        "/search",
        "/video",
        "/foto"
    ]

    if path in excluded:
        return False

    if path.endswith("/"):
        return False

    return True

### Mengambil URL Finance

In [42]:
finance_urls_test = get_article_links(
    "https://finance.detik.com/",
    "finance.detik.com"
)

finance_urls_test = [
    url
    for url in finance_urls_test
    if is_finance_article(url)
]

finance_urls_test = list(
    dict.fromkeys(finance_urls_test)
)

print(
    "URL Finance:",
    len(finance_urls_test)
)

URL Finance: 86


### Fungsi pagination Finance

In [43]:
def get_finance_index_url(page):

    if page == 1:
        return "https://finance.detik.com/indeks"

    return f"https://finance.detik.com/indeks?page={page}"

### Tes pagination

In [44]:
for page in range(1, 6):
    print(get_finance_index_url(page))

https://finance.detik.com/indeks
https://finance.detik.com/indeks?page=2
https://finance.detik.com/indeks?page=3
https://finance.detik.com/indeks?page=4
https://finance.detik.com/indeks?page=5


### Mengumpulkan URL Finance

In [45]:
def collect_finance_urls(
    max_pages=10,
    target_urls=150
):

    all_urls = set()

    for page in range(1, max_pages + 1):

        index_url = get_finance_index_url(page)

        print(
            f"Mengambil indeks Finance halaman {page}"
        )

        urls = get_article_links(
            index_url,
            "finance.detik.com"
        )

        for url in urls:

            if is_finance_article(url):
                all_urls.add(url)

        print(
            f"Total URL unik: {len(all_urls)}"
        )

        if len(all_urls) >= target_urls:
            break

        time.sleep(1)

    return list(all_urls)

### Menjalankan pengumpulan sport

In [46]:
finance_urls = collect_finance_urls(
    max_pages=10,
    target_urls=150
)

Mengambil indeks Finance halaman 1
Total URL unik: 36
Mengambil indeks Finance halaman 2
Total URL unik: 56
Mengambil indeks Finance halaman 3
Total URL unik: 76
Mengambil indeks Finance halaman 4
Total URL unik: 94
Mengambil indeks Finance halaman 5
Total URL unik: 114
Mengambil indeks Finance halaman 6
Total URL unik: 134
Mengambil indeks Finance halaman 7
Total URL unik: 153


In [47]:
print(
    "Total URL Finance:",
    len(finance_urls)
)

Total URL Finance: 153


### Crawling 100 Finance

In [48]:
finance_data = crawl_articles(
    finance_urls,
    label="finance",
    target=100
)

[finance] 1/100 -> https://finance.detik.com/moneter/d-8654026/jurus-bank-perkuat-likuiditas-bikin-pertumbuhan-bisnis-tetap-sehat
  -> Berhasil
[finance] 2/100 -> https://finance.detik.com/loker
  -> Gagal mendapatkan isi
[finance] 2/100 -> https://finance.detik.com/berita-ekonomi-bisnis/d-8654222/purbaya-ngaku-tak-pusingkan-anggaran-bgn-tahun-ini-tak-sampai-rp-200-t
  -> Berhasil
[finance] 3/100 -> https://finance.detik.com/energi/d-8654238/pertamina-waspada-solar-subsidi-bocor-selisih-harga-tembus-rp-18-200
  -> Berhasil
[finance] 4/100 -> https://finance.detik.com/berita-ekonomi-bisnis/d-8653186/17-perjalanan-krl-dibatalkan-imbas-perawatan-armada-buatan-inka
  -> Berhasil
[finance] 5/100 -> https://finance.detik.com/berita-ekonomi-bisnis/d-8655674/prabowo-janji-berikan-bonus-rp-3-m-buat-peraih-emas-asian-games-2026
  -> Berhasil
[finance] 6/100 -> https://finance.detik.com/moneter/d-8653044/pinjaman-online-warga-ri-makin-banyak-tembus-rp-105-triliun
  -> Berhasil
[finance] 7/100 -> 

In [49]:
print(
    "Jumlah data Finance:",
    len(finance_data)
)

Jumlah data Finance: 100


### Gabungkan Data

In [50]:
data = sport_data + finance_data

In [51]:
df = pd.DataFrame(data)

### Buat ID ulang

In [52]:
df["id"] = range(1, len(df) + 1)

In [54]:
df = df[
    [
        "id",
        "isi_berita",
        "label"
    ]
]

In [55]:
print("Jumlah data:", len(df))

Jumlah data: 200


### Cek jumlah setiap label

In [56]:
print(
    df["label"].value_counts()
)

label
sport      100
finance    100
Name: count, dtype: int64


### Cek data kosong

In [57]:
print(df.isnull().sum())

id            0
isi_berita    0
label         0
dtype: int64


### Cek duplikasi

In [58]:
print(
    "Duplikat isi berita:",
    df["isi_berita"].duplicated().sum()
)

Duplikat isi berita: 8


### Menghapus data duplikat

In [59]:
df = df.drop_duplicates(
    subset=["isi_berita"]
).reset_index(drop=True)

df["id"] = range(1, len(df) + 1)

In [74]:
print(df["label"].value_counts())

label
sport      100
finance     92
Name: count, dtype: int64


### Simpan Ke csv

In [75]:
df.to_csv(
    "data/dataset_200.csv",
    index=False,
    encoding="utf-8-sig"
)

In [76]:
df_check = pd.read_csv(
    "data/dataset_200.csv"
)

In [77]:
print(df_check.shape)

(192, 3)


In [78]:
df_check.head()

,id,isi_berita,label
0,1,Jakarta - Sebanyak 432 atlet membela Indonesia...,sport
1,2,FIA World Endurance Championship 2026 memasuki...,sport
2,3,Campus League Badminton Regional Semarang tela...,sport
3,4,Kontingen Indonesia bersiap menatap Asian Game...,sport
4,5,"Jelang Asian Games 2026, Timnas basket 3x3 put...",sport


In [80]:
print("Jumlah data saat ini:", len(df_check))
print("\nDistribusi label:")
print(df_check["label"].value_counts())

Jumlah data saat ini: 192

Distribusi label:
label
sport      100
finance     92
Name: count, dtype: int64


### Data tambahan untuk finance

In [84]:
existing_finance_texts = set(
    df_check.loc[
        df_check["label"] == "finance",
        "isi_berita"
    ]
    .dropna()
    .str.strip()
)

print(
    "Finance unik yang sudah ada:",
    len(existing_finance_texts)
)

Finance unik yang sudah ada: 92


In [85]:
print("Jumlah URL Finance yang tersedia:", len(finance_urls))

Jumlah URL Finance yang tersedia: 153


### Fungsi crawl tambahan untuk finance

In [86]:
def crawl_additional_finance(
    urls,
    existing_texts,
    target=8
):
    
    additional_data = []
    
    for url in urls:
        
        if len(additional_data) >= target:
            break
        
        print(
            f"Memeriksa: "
            f"{len(additional_data)}/{target} "
            f"-> {url}"
        )
        
        text = extract_article(url)
        
        if text is None:
            print("  -> Gagal mendapatkan isi")
            continue
        
        text = text.strip()
        
        # Cek apakah isi sudah pernah ada
        if text in existing_texts:
            print("  -> Duplikat, dilewati")
            continue
        
        # Pastikan artikel belum ditambahkan
        if any(
            item["isi_berita"] == text
            for item in additional_data
        ):
            print("  -> Duplikat tambahan, dilewati")
            continue
        
        additional_data.append({
            "isi_berita": text,
            "label": "finance"
        })
        
        print("  -> Berhasil, artikel baru")
        
        time.sleep(1)
    
    return additional_data

### Crawl data tambahan

In [87]:
finance_tambahan = crawl_additional_finance(
    finance_urls,
    existing_finance_texts,
    target=8
)

Memeriksa: 0/8 -> https://finance.detik.com/moneter/d-8654026/jurus-bank-perkuat-likuiditas-bikin-pertumbuhan-bisnis-tetap-sehat
  -> Duplikat, dilewati
Memeriksa: 0/8 -> https://finance.detik.com/loker
  -> Gagal mendapatkan isi
Memeriksa: 0/8 -> https://finance.detik.com/berita-ekonomi-bisnis/d-8654222/purbaya-ngaku-tak-pusingkan-anggaran-bgn-tahun-ini-tak-sampai-rp-200-t
  -> Duplikat, dilewati
Memeriksa: 0/8 -> https://finance.detik.com/energi/d-8654238/pertamina-waspada-solar-subsidi-bocor-selisih-harga-tembus-rp-18-200
  -> Duplikat, dilewati
Memeriksa: 0/8 -> https://finance.detik.com/berita-ekonomi-bisnis/d-8653186/17-perjalanan-krl-dibatalkan-imbas-perawatan-armada-buatan-inka
  -> Duplikat, dilewati
Memeriksa: 0/8 -> https://finance.detik.com/berita-ekonomi-bisnis/d-8655674/prabowo-janji-berikan-bonus-rp-3-m-buat-peraih-emas-asian-games-2026
  -> Duplikat, dilewati
Memeriksa: 0/8 -> https://finance.detik.com/moneter/d-8653044/pinjaman-online-warga-ri-makin-banyak-tembus-rp-10

In [88]:
print(
    "Jumlah Finance tambahan:",
    len(finance_tambahan)
)

Jumlah Finance tambahan: 8


In [89]:
finance_tambahan_df = pd.DataFrame(
    finance_tambahan
)

In [90]:
df_final = pd.concat(
    [df_check, finance_tambahan_df],
    ignore_index=True
)

In [91]:
df_final["id"] = range(
    1,
    len(df_final) + 1
)

In [92]:
df_final = df_final[
    [
        "id",
        "isi_berita",
        "label"
    ]
]

In [93]:
print("Jumlah data:", len(df_final))

print("\nDistribusi label:")
print(df_final["label"].value_counts())

Jumlah data: 200

Distribusi label:
label
sport      100
finance    100
Name: count, dtype: int64


In [94]:
print(
    "Jumlah duplikat:",
    df_final["isi_berita"].duplicated().sum()
)

Jumlah duplikat: 0


In [95]:
print("\nData kosong:")
print(df_final.isnull().sum())


Data kosong:
id            0
isi_berita    0
label         0
dtype: int64


In [96]:
df_final.to_csv(
    "data/dataset_200.csv",
    index=False,
    encoding="utf-8-sig"
)

In [97]:
df_check = pd.read_csv(
    "data/dataset_200.csv"
)

print("Shape:", df_check.shape)
print("\nDistribusi:")
print(df_check["label"].value_counts())

Shape: (200, 3)

Distribusi:
label
sport      100
finance    100
Name: count, dtype: int64


In [98]:
df_check.head()

,id,isi_berita,label
0,1,Jakarta - Sebanyak 432 atlet membela Indonesia...,sport
1,2,FIA World Endurance Championship 2026 memasuki...,sport
2,3,Campus League Badminton Regional Semarang tela...,sport
3,4,Kontingen Indonesia bersiap menatap Asian Game...,sport
4,5,"Jelang Asian Games 2026, Timnas basket 3x3 put...",sport
